In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# ------------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------------
DATA_DIR = '/kaggle/input/competitions/domestic-flight-price-prediction-challenge'

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')

test_ids = test['id'].copy()

train = train.dropna(subset=['price']).reset_index(drop=True)

print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("\nTrain null counts:\n", train.isnull().sum())
print("\nTest null counts:\n", test.isnull().sum())   # (stray "." removed)

# ------------------------------------------------------------------
# 2. Configuration
# ------------------------------------------------------------------
STOP_MAP = {'zero': 0, 'one': 1, 'two_or_more': 2}
CLASS_MAP = {'Economy': 0, 'Business': 1}

BASE_CAT_COLS = [
    'airline', 'source_city', 'destination_city',
    'departure_time', 'arrival_time', 'flight'
]


def get_prefix(df):
    return df['flight'].astype('string').str.split().str[0]


# ------------------------------------------------------------------
# 3. Learn imputation values from training data only
# ------------------------------------------------------------------
def fit_preprocessor(df):
    p = {}

    tmp = pd.DataFrame({
        'prefix': get_prefix(df),
        'airline': df['airline']
    }).dropna()

    p['airline_map'] = (
        tmp.groupby('prefix')['airline']
        .agg(lambda s: s.mode().iloc[0])
        .to_dict()
    )

    stops = df['stops'].map(STOP_MAP)
    p['stops_mode'] = stops.mode().iloc[0]

    classes = df['flight_class'].map(CLASS_MAP)
    p['class_mode'] = classes.mode().iloc[0]

    stops = stops.fillna(p['stops_mode'])

    p['duration_by_stops'] = (
        df.assign(stops_num=stops)
        .groupby('stops_num')['duration']
        .median()
        .to_dict()
    )

    p['duration_median'] = df['duration'].median()
    p['days_left_median'] = df['days_left'].median()

    p['cat_modes'] = {c: df[c].mode().iloc[0] for c in BASE_CAT_COLS}

    return p


# ------------------------------------------------------------------
# 4. Clean data and create features
# ------------------------------------------------------------------
def clean(df, p):
    df = df.copy()

    prefix = get_prefix(df)

    df['airline'] = df['airline'].fillna(prefix.map(p['airline_map']))
    df['airline'] = df['airline'].fillna(p['cat_modes']['airline'])

    df['stops'] = df['stops'].map(STOP_MAP).fillna(p['stops_mode'])
    df['flight_class'] = df['flight_class'].map(CLASS_MAP).fillna(p['class_mode'])

    df['duration'] = df['duration'].fillna(df['stops'].map(p['duration_by_stops']))
    df['duration'] = df['duration'].fillna(p['duration_median'])

    df['days_left'] = df['days_left'].fillna(p['days_left_median'])

    for col, mode_value in p['cat_modes'].items():
        df[col] = df[col].fillna(mode_value)

    df['flight_prefix'] = get_prefix(df).fillna('Unknown').astype(str)

    df['route'] = (
        df['source_city'].astype(str) + '_to_' + df['destination_city'].astype(str)
    )
    df['airline_route'] = df['airline'].astype(str) + '_on_' + df['route']

    df['days_left_squared'] = df['days_left'] ** 2
    df['duration_x_stops'] = df['duration'] * df['stops']
    df['days_left_x_business'] = df['days_left'] * (df['flight_class'] == 1).astype(int)

    df = df.drop(columns=['id'], errors='ignore')
    return df


params = fit_preprocessor(train)

train_clean = clean(train, params)
test_clean = clean(test, params)

y = train_clean['price'].copy()
X_all = train_clean.drop(columns=['price'])
X_test_all = test_clean.copy()

# ------------------------------------------------------------------
# 5. One-hot encode categorical columns
# ------------------------------------------------------------------
CAT_COLS = [
    'airline', 'source_city', 'destination_city',
    'departure_time', 'arrival_time', 'flight_prefix',
    'route', 'airline_route', 'flight'
]

X_enc = pd.get_dummies(X_all, columns=CAT_COLS, dtype=np.uint8)
X_test_enc = pd.get_dummies(X_test_all, columns=CAT_COLS, dtype=np.uint8)
X_test_enc = X_test_enc.reindex(columns=X_enc.columns, fill_value=0)

assert list(X_enc.columns) == list(X_test_enc.columns)
assert X_enc.isnull().sum().sum() == 0
assert X_test_enc.isnull().sum().sum() == 0

print("\nEncoded train shape:", X_enc.shape)
print("Encoded test shape:", X_test_enc.shape)

# ------------------------------------------------------------------
# 6. 5-Fold cross-validation with XGBoost
# ------------------------------------------------------------------
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X_enc))                # out-of-fold predictions
test_pred = np.zeros(len(X_test_enc))          # averaged test predictions
importances = np.zeros(X_enc.shape[1])
fold_scores = []
best_iters = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_enc), start=1):
    print(f"\n========== Fold {fold}/{N_SPLITS} ==========")

    X_tr, X_val = X_enc.iloc[tr_idx], X_enc.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = XGBRegressor(
        objective='reg:squarederror',
        eval_metric='rmse',
        n_estimators=3000,
        learning_rate=0.05,
        max_depth=8,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=5,
        tree_method='hist',
        early_stopping_rounds=150,
        random_state=42 + fold,
        n_jobs=-1,
        verbosity=1
    )

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=200
    )

    # Validation predictions for this fold
    val_pred = model.predict(X_val)
    oof_pred[val_idx] = val_pred

    mae = mean_absolute_error(y_val, val_pred)
    rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    r2 = r2_score(y_val, val_pred)
    fold_scores.append((mae, rmse, r2))
    best_iters.append(model.best_iteration)

    print(f"Fold {fold} -> MAE: {mae:.2f} | RMSE: {rmse:.2f} | R2: {r2:.4f} "
          f"| best_iter: {model.best_iteration}")

    # Accumulate test predictions and feature importances
    test_pred += model.predict(X_test_enc) / N_SPLITS
    importances += model.feature_importances_ / N_SPLITS

# ------------------------------------------------------------------
# 7. Cross-validation summary
# ------------------------------------------------------------------
scores = np.array(fold_scores)

print("\n--- 5-Fold CV Performance (mean ± std across folds) ---")
print(f"MAE : {scores[:, 0].mean():.2f} ± {scores[:, 0].std():.2f}")
print(f"RMSE: {scores[:, 1].mean():.2f} ± {scores[:, 1].std():.2f}")
print(f"R2  : {scores[:, 2].mean():.4f} ± {scores[:, 2].std():.4f}")
print("Best iterations per fold:", best_iters)

print("\n--- Overall Out-of-Fold Performance ---")
print(f"MAE : {mean_absolute_error(y, oof_pred):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y, oof_pred)):.2f}")
print(f"R2  : {r2_score(y, oof_pred):.4f}")

# ------------------------------------------------------------------
# 8. Feature importance (averaged over folds)
# ------------------------------------------------------------------
importance = pd.DataFrame({
    'feature': X_enc.columns,
    'importance': importances
}).sort_values('importance', ascending=False)

print("\n--- Top 20 Features ---")
print(importance.head(20).to_string(index=False))

# ------------------------------------------------------------------
# 9. Save submission (average of the 5 fold models)
# ------------------------------------------------------------------
test_pred = np.clip(test_pred, 0, None)   # prices cannot be negative

submission = pd.DataFrame({
    'id': test_ids,
    'price': test_pred
})

submission.to_csv('submission_xgboost_5fold.csv', index=False)

print("\nSaved submission_xgboost_5fold.csv")
print(submission.shape)
print(submission.head())